# 1. What this changes
Run on the developer's machine **after the deployed `/chat` baseline succeeds**. The runtime remains in App Service. This notebook creates/reuses an Entra blueprint, blueprint principal and one child identity; trusts the **already attached** user-assigned managed identity (UAMI); registers the child in Agent 365; and links an External Agent in Foundry. No Graph/Blob workload grants, OBO, secrets, AI Gateway, agent user, or second web app.

Administrative Graph consent belongs to the **provisioning public client**, never the running agent. `SETUP`, `REGISTER`, and `LINK` explicitly authorize separate writes. Azure CLI is used only for Azure/Foundry administration, not Agent ID Graph sign-in. Ignored `.local` state binds all retries to the tenant, web app and provisioning client. Do not delete it after an interrupted write.

**Preview warning:** identity creation uses Graph v1.0; typed FIC operations use `/beta`; `/beta/copilot/agentRegistrations` is an **undocumented preview contract**, checked against Microsoft Agent365-devTools source on 2026-11-23. Unsupported contracts stop; there is no legacy endpoint or delegated runtime fallback.

# 2. Prerequisites
Read README administrative prerequisites and current linked Microsoft terms. Your tenant needs Agent ID/Agent 365 eligibility, applicable Agent 365 licensing, preview access and accepted terms. An administrator must preconfigure a dedicated single-tenant **public client** with Mobile and desktop redirect **`http://localhost:8400`**, enable public-client flows if required by tenant policy, add the delegated permissions below and explicitly grant consent. This notebook never adds consent or elevates your role.

| Operation | Delegated Graph permission on provisioning client |
| --- | --- |
| Create/read blueprint | `AgentIdentityBlueprint.Create`, `AgentIdentityBlueprint.Read.All` |
| Create/read blueprint principal | `AgentIdentityBlueprintPrincipal.Create`, `AgentIdentityBlueprintPrincipal.Read.All` |
| Create/read child and sponsors | `AgentIdentity.Create.All`, `AgentIdentity.Read.All` |
| Create/read exact FIC on a blueprint you own | `AgentIdentityBlueprint.Create`, `AgentIdentityBlueprint.Read.All` (beta contract) |
| Register/read Agent 365 | `AgentRegistration.ReadWrite.All`, plus identity/user reads above/below |
| Resolve a real existing human sponsor | `User.ReadBasic.All` |

The least-privilege path creates objects owned by the setup caller; owners can manage their blueprint's children. Nonowners need Agent ID Administrator (Agent ID Developer is supported for blueprint/principal creation, not a blanket grant for all operations). Reused objects require appropriate ownership or role authority. Credential-management alternatives such as `AgentIdentityBlueprint.AddRemoveCreds.All` are **not silently requested**. See README for operation-specific references.

For Foundry: setup caller needs **Foundry User** on the target project, plus read access to its connection and the linked App Insights resource. Trace viewers need **Reader or Monitoring Reader** on that resource. The App Service system identity needs model access; Entra-authenticated Azure Monitor ingestion needs **Monitoring Metrics Publisher** on App Insights. None of these are Graph permissions.

Select this repo's `.venv` kernel. Complete browser PKCE sign-in on the kernel machine within 180 seconds; port 8400 must be free. Remote/headless kernels need a secured browser/loopback arrangement and are not automatically supported. Never share sign-in URLs, token/cache files, profiles, or notebook outputs.

# 3. Configuration and authentication
Copy deployment outputs, not identity values guessed by display name. Set notebook-only values in `.env`. No tokens are persisted. Primary mode is dual export; `AGENT365_ONLY` must be explicitly selected to opt out.

In [ ]:
import json
import os
import sys
from pathlib import Path

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
if not (root / 'src' / 'onboarding.py').is_file():
    raise RuntimeError('Open this notebook from the starter repository.')
sys.path.insert(0, str(root))
from dotenv import load_dotenv
from src import onboarding as setup
from src.observability import project_endpoint
from src.telemetry_auth import guid

load_dotenv(root / '.env')
AGENT365_ONLY = False  # Explicit opt-out only; never inferred from missing settings.
deployment = json.loads((root / '.local' / 'deployment.json').read_text(encoding='utf-8-sig'))
setup.validate_deployment(deployment)
setup.verify_azure_deployment(deployment)
tenant_id = deployment['tenantId']
client_id = guid(os.environ.get('ENTRA_PROVISIONING_CLIENT_ID'), 'ENTRA_PROVISIONING_CLIENT_ID')
sponsor_id = guid(os.environ.get('ONBOARDING_SPONSOR_USER_ID'), 'ONBOARDING_SPONSOR_USER_ID')
target_endpoint = project_endpoint(os.getenv('FOUNDRY_TELEMETRY_PROJECT_ENDPOINT') or deployment['foundryProjectEndpoint'])
external_name = os.getenv('FOUNDRY_EXTERNAL_AGENT_NAME', 'AppServiceLearningAgentExternal')
state_path = root / '.local' / 'onboarding-state.json'
state = setup.load_state(state_path, deployment, client_id)
print('Target web app:', deployment['webAppName'])
print('Destinations:', 'Agent 365 ONLY; Foundry not configured' if AGENT365_ONLY else 'Agent 365 AND Foundry External Agent')
if target_endpoint != deployment['foundryProjectEndpoint']:
    print('Explicit DIFFERENT telemetry project:', target_endpoint)
graph = setup.sign_in(tenant_id, client_id)


# 4. Create/reuse blueprint and blueprint principal
The helper uses explicit typed requests: `POST /v1.0/applications/microsoft.graph.agentIdentityBlueprint`, then `POST /v1.0/servicePrincipals/microsoft.graph.agentIdentityBlueprintPrincipal`. It reads before creating and checks `appId`. Names for new objects include a stable deployment hash, not a global generic name. Optional explicit IDs reuse existing objects; ambiguous matches and mismatched state stop. `managerApplications` identifies this provisioning client, not the CLI's first-party app. No unrelated blueprint is patched.

In [ ]:
if input('Type SETUP to create/reuse identities and exact federated trust: ').strip() != 'SETUP':
    raise RuntimeError('Setup cancelled; no setup writes authorized.')
setup_confirmed = True
setup.ensure_blueprint(
    graph, state, state_path, sponsor_id,
    blueprint_id=os.getenv('ENTRA_BLUEPRINT_OBJECT_ID') or None,
    child_id=os.getenv('ENTRA_AGENT_IDENTITY_OBJECT_ID') or None,
)
print('Blueprint and its principal readback verified. IDs retained in ignored state.')


# 5. Create/reuse one child identity and resolve its real sponsor
`POST /v1.0/servicePrincipals/microsoft.graph.agentIdentity` supplies the blueprint **client ID** and selected existing sponsor user. Reuse reads the typed child and its sponsor relationship. The selected sponsor must resolve through `/users/{id}`; no administrator profile or synthetic user is substituted. Runtime sponsor attribution is accountability metadata, **not an authenticated HTTP caller**.

In [ ]:
if not globals().get('setup_confirmed'):
    raise RuntimeError('Run the SETUP confirmation first.')
setup.ensure_child(graph, state, state_path, sponsor_id)
print('Child identity and existing human sponsor verified; profile is not displayed.')


# 6. Configure federation on the existing blueprint
Read `/beta/applications/{blueprintObjectId}/microsoft.graph.agentIdentityBlueprint/federatedIdentityCredentials` before creating. Exact trust is issuer `https://login.microsoftonline.com/{tenantId}/v2.0`, subject = attached UAMI **principal/object ID**, audience = `api://AzureADTokenExchange`. A name match alone is insufficient. The runtime selects that same UAMI by its **client ID**. The App Service system identity remains responsible for model access; this is the sample's selected blueprint-federation path, not a universal ban on system identities.

In [ ]:
if not globals().get('setup_confirmed'):
    raise RuntimeError('Run the SETUP confirmation first.')
setup.ensure_federation(graph, state, state_path)
print('Exact federated credential readback verified.')


# 7. Register the existing child with Agent 365
This is distinct from merely creating an Entra identity. The **undocumented preview** `POST /beta/copilot/agentRegistrations` binds `agentIdentityId`, `sourceAgentId`, blueprint client ID and this provisioning client's `managedByAppId`. A request GUID is persisted **before** the write. GET readback must match; 401/403/network errors never mean absence. Interrupted/ambiguous writes probe the same request (including the service's `T_` form) and otherwise stop for reconciliation. Never delete state to force a new registration.

In [ ]:
if input('Type REGISTER to register/read back the existing child with Agent 365: ').strip() != 'REGISTER':
    raise RuntimeError('Registration cancelled.')
setup.ensure_registration(graph, state, state_path)
print('Agent 365 registration readback verified. Runtime export/portal visibility NOT yet verified.')


# 8. Link a Foundry External Agent and verify the connected destination
| Component | Purpose |
| --- | --- |
| Agent 365 registration | Identifies/authorizes the child for this Agent 365 integration |
| Foundry External Agent record | Matches incoming spans by `otel_agent_id = child GUID` |
| Azure Monitor exporter | Actually sends those spans to the project's linked App Insights |

Neither registration replaces the other. External registration does not deploy/invoke code, proxy `/chat`, configure trust, or grant export access. Setup uses Azure CLI user credentials **outside** the hosted app and the pinned SDK `ExternalAgentDefinition`, with `allow_preview=True`. Read first; only a genuine 404 permits creation; mismatches stop. An ambiguous create reconciles instead of blindly creating versions.

Connection verification compares the project's telemetry API connection, the supplied connection string, and the selected ARM resource's instrumentation key. If you cannot inspect them, mark the connection **unverified** and ask an authorized person to check Foundry **Manage > Project details > Connected resources** and the resource's **Overview**. Do not enable dual export while unverified. Never print a connection string.

In [ ]:
if AGENT365_ONLY:
    print('Foundry External Agent observability NOT CONFIGURED (explicit Agent 365-only mode).')
else:
    if input('Type LINK to create/reuse the matching Foundry External Agent: ').strip() != 'LINK':
        raise RuntimeError('Foundry linking cancelled; dual export cannot be activated.')
    from azure.ai.projects import AIProjectClient
    from azure.identity import AzureCliCredential
    resource_id = os.environ.get('APPLICATIONINSIGHTS_RESOURCE_ID', '').strip()
    connection_string = os.environ.get('APPLICATIONINSIGHTS_CONNECTION_STRING', '').strip()
    with AzureCliCredential(tenant_id=tenant_id) as credential:
        with AIProjectClient(endpoint=target_endpoint, credential=credential, allow_preview=True) as project:
            setup.ensure_external(project, external_name, state['childAgentId'], state, state_path, target_endpoint)
            state['foundry']['applicationInsightsResourceId'] = resource_id
            state['foundry']['connectionVerified'] = False
            setup.save_private(state_path, state)
            try:
                setup.verify_insights(project, resource_id, connection_string)
            except Exception:
                raise RuntimeError('Linked resource UNVERIFIED. Check Manage > Project details > Connected resources and App Insights Overview with an authorized reader. No dual-export settings generated.') from None
            state['foundry']['connectionVerified'] = True
            setup.save_private(state_path, state)
    print('Foundry external readback and linked destination verified; ingested traces remain pending.')


# 9. Prepare protected runtime settings
The ignored artifact contains sponsor metadata and the connection string; treat it as protected configuration, not a notebook output. Do not display it or commit execution outputs. Default settings activate both destinations with content capture **off**, offline spooling **off**, and explicit system-identity Azure Monitor authentication. Role assignment remains an explicit administrator/deployment action.

In [ ]:
artifact_path = root / '.local' / 'runtime-settings.json'
setup.prepare_runtime_settings(
    state, artifact_path, agent365_only=AGENT365_ONLY,
    connection_string='' if AGENT365_ONLY else connection_string,
)
print('Protected .local/runtime-settings.json prepared; no settings values displayed.')
graph.close()


# 10. Activate, invoke, and verify
From the repository root in PowerShell:
```powershell
.\scripts\enable-observability.ps1 -SettingsFile .local\runtime-settings.json -Approve -AssignMonitoringRole
.\scripts\verify.ps1 -DeploymentFile .local\deployment.json -SettingsFile .local\runtime-settings.json
```
For the explicitly selected opt-out replace `-AssignMonitoringRole` with `-Agent365Only` in activation, and add `-Agent365Only` to verification. Readback uses bounded administrative browser sign-in again; the hosted runtime never signs in interactively. Activation changes app settings and restarts, not running source files.

`src/observability.py` initializes one pipeline before framework imports. A request obtains/caches the app-only token through UAMI -> blueprint FMI -> child, sets tenant/child/blueprint/conversation baggage, and emits invocation, inference and output metadata. The same trace reaches both exporters; Foundry's `otel_agent_id` equals the root span's child GUID. Token/export outages are diagnosed without blocking a successful model reply.

Verify each outcome separately: deployment/model response; both registration readbacks; app-only token acquisition; transport/export acceptance; Agent 365 Activity indexing; and **Build > Agents > this external agent > Traces** with the same trace ID and child GUID. Expect approximately 2-5 minutes or longer for indexing. `force_flush`, token acquisition, raw App Insights data and HTTP 200 are **not** proof of agent-scoped portal visibility. All portal outcomes remain **PENDING until a human confirms them**. See README verification queries, partial-success limitations, troubleshooting and targeted cleanup.